In [ ]:
# ===========================================
# MODELO DE LENGUAJE BIGRAMA CON SUAVIZADO DE LAPLACE
# Procesamiento del Lenguaje Natural - Profesor Carlos Ogando
# likelihood en log y generación con muestreo
# ===========================================

from collections import Counter
import pandas as pd
import math
import random


# Datos de Entrenamiento

training_sentences = [
    "The cat sits on the mat.",
    "My dog sits on the mat.",
    "His cat sleeps on the mat.",
    "My dog sleeps on his mat.",
    "The cat jumps over her mat.",
    "The dog jumps over the mat.",
    "The cat runs across the yard.",
    "The dog runs across the yard.",
    "The cat plays with the ball.",
    "Her dog plays with the ball.",
    "His cat chases my dog.",
    "My dog chases the cat.",
    "His cat hides under the table.",
    "My dog hides under my table.",
    "The cat and dog sit together."
]


# Preprocesamiento

train_data = []
for sentence in training_sentences:
    words = sentence.lower().replace(".", "").split()
    train_data.append(["<START>"] + words + ["STOP"])

# Contar unigramas y bigramas
unigrams = Counter()
bigrams = Counter()
for words in train_data:
    unigrams.update(words)
    bigrams.update(zip(words[:-1], words[1:]))

V = len(unigrams)  # tamaño del vocabulario

# Probabilidades suavizadas (Laplace)
prob_bigram = {}
for (w1, w2), count in bigrams.items():
    prob_bigram[(w1, w2)] = (count + 1) / (unigrams[w1] + V)

# Crear tabla de bigramas
df_bigrams = pd.DataFrame([
    {"Bigrama": f"{w1} | {w2}", "Probabilidad": prob_bigram[(w1, w2)]}
    for (w1, w2) in prob_bigram
])

print("=== Tabla de Bigramas con Probabilidades ===")
print(df_bigrams.head(15))
print()


# Likelihood de Datos de Prueba

test_sentences = [
    "My cat plays with her dog.",
    "Her dog runs across my yard.",
    "His dog sleeps on my mat."
]

def sentence_likelihood_log(sentence):
    words = ["<START>"] + sentence.lower().replace(".", "").split() + ["STOP"]
    log_L = 0.0
    for w1, w2 in zip(words[:-1], words[1:]):
        count_bigram = bigrams.get((w1, w2), 0)
        count_w1 = unigrams.get(w1, 0)
        P = (count_bigram + 1) / (count_w1 + V)
        log_L += math.log(P)
    return math.exp(log_L)  # regresamos likelihood real, no 0

print("=== Likelihood de Oraciones de Prueba (corregido) ===")
for s in test_sentences:
    print(f"{s} → {sentence_likelihood_log(s):.12f}")
print()


# Generación de Texto (Muestreo Probabilístico)
def generate_sentence_sampling(start_word, max_len=20):
    sentence = [start_word]
    w1 = start_word
    for _ in range(max_len):
        candidates = {}
        for w2 in unigrams:
            count_bigram = bigrams.get((w1, w2), 0)
            count_w1 = unigrams.get(w1, 0)
            candidates[w2] = (count_bigram + 1) / (count_w1 + V)
        # Muestreo probabilístico
        words_list = list(candidates.keys())
        probs = list(candidates.values())
        w2 = random.choices(words_list, weights=probs, k=1)[0]
        if w2 == "STOP":
            break
        sentence.append(w2)
        w1 = w2
    return " ".join(sentence)

print("=== Oraciones Generadas (muestreo) ===")
for start in ["my", "her", "the"]:
    print(f"{start} → {generate_sentence_sampling(start)}")
print()


#Perplexity

def perplexity(sentence):
    words = ["<START>"] + sentence.lower().replace(".", "").split() + ["STOP"]
    log_sum = 0
    N = len(words)
    for w1, w2 in zip(words[:-1], words[1:]):
        count_bigram = bigrams.get((w1, w2), 0)
        count_w1 = unigrams.get(w1, 0)
        P = (count_bigram + 1) / (count_w1 + V)
        log_sum += math.log(P)
    return math.exp(-log_sum / N)

# Calcular perplejidad para test
print("=== Perplexity de Datos de Prueba ===")
pps = []
for s in test_sentences:
    pp = perplexity(s)
    pps.append(pp)
    print(f"{s} → {pp:.4f}")

print(f"\nPerplexity promedio (test): {sum(pps)/len(pps):.4f}")


=== Tabla de Bigramas con Probabilidades ===
          Bigrama  Probabilidad
0   <START> | the      0.190476
1       the | cat      0.159091
2      cat | sits      0.055556
3       sits | on      0.103448
4        on | the      0.129032
5       the | mat      0.113636
6      mat | STOP      0.212121
7    <START> | my      0.119048
8        my | dog      0.181818
9      dog | sits      0.055556
10  <START> | his      0.095238
11      his | cat      0.129032
12   cat | sleeps      0.055556
13    sleeps | on      0.103448
14   dog | sleeps      0.055556

=== Likelihood de Oraciones de Prueba (corregido) ===
My cat plays with her dog. → 0.000000002739
Her dog runs across my yard. → 0.000000002040
His dog sleeps on my mat. → 0.000000003661

=== Oraciones Generadas (muestreo) ===
my → my dog and sit sit cat table sleeps under table hides the hides on together dog hides table together the mat
her → her and hides <START> across her my hides sits table hides the with and <START> the mat
the → t

In [ ]:
df_bigrams.to_csv("tabla_bigramas.csv", index=False)